# 06 · Contraction with einsum / Contracción con einsum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb)

*Part IV · exercise · 15 min*

This notebook teaches one rule that looks compact in code but has a very simple meaning:

> **If an index disappears after `->`, NumPy sums over it. If the index remains, it survives in the output.**

We will use that one rule on real microscopy pixels, real handwritten-digit pixels, matrix operations, and image retrieval.

> 🇪🇸 Este cuaderno enseña una regla que parece compacta en el código, pero tiene un significado muy sencillo:
>
> **Si un índice desaparece después de `->`, NumPy suma sobre él. Si el índice permanece, sobrevive en la salida.**
>
> Usaremos esa misma regla sobre píxeles reales de microscopía, píxeles reales de dígitos manuscritos, operaciones matriciales y búsqueda de imágenes.

## What you will be able to do / Lo que podrás hacer

- Explain **contraction** and `einsum` without memorising formulas.
- Read `hwc,c->hw`, `ik,kj->ij`, and `id,jd->ij` as sentences.
- Convert a real RGB microscopy image to grayscale with one contraction.
- Understand trace, transpose, and matrix multiplication through index movement.
- Compute all **3,229,209 pairwise similarities** between 1,797 real handwritten digits.
- Explore how cosine similarity and raw dot product produce different neighbours.

> 🇪🇸
>
> - Explicar **contracción** y `einsum` sin memorizar fórmulas.
> - Leer `hwc,c->hw`, `ik,kj->ij` e `id,jd->ij` como frases.
> - Convertir una imagen RGB real de microscopía a escala de grises con una contracción.
> - Entender traza, transposición y producto matricial mediante el movimiento de índices.
> - Calcular las **3.229.209 similitudes por pares** entre 1.797 dígitos manuscritos reales.
> - Explorar cómo similitud coseno y producto punto producen vecinos diferentes.

## How to read `einsum` / Cómo leer `einsum`

Think of the letters as **axis names**, not mysterious algebra.

| Expression / Expresión | What disappears? / ¿Qué desaparece? | Output meaning / Significado de salida |
|---|---|---|
| `hwc,c->hw` | `c` | one value per `(h,w)` pixel / un valor por píxel `(h,w)` |
| `ii->` | `i` | one scalar / un escalar |
| `ij->ji` | nothing / nada | same values, axes reversed / mismos valores, ejes invertidos |
| `ik,kj->ij` | `k` | matrix product / producto matricial |
| `id,jd->ij` | `d` | one score for each pair `(i,j)` / una puntuación por cada par `(i,j)` |

### The sentence to remember / La frase para recordar

> **Before the arrow = available indices. After the arrow = indices you want to keep. Missing indices are summed.**

> 🇪🇸
>
> **Antes de la flecha = índices disponibles. Después de la flecha = índices que quieres conservar. Los índices que desaparecen se suman.**

## Setup / Preparación

We use two real datasets already included in standard Python libraries:

- a real colour microscopy image from `skimage.data`;
- the 1,797-image handwritten-digits dataset from `sklearn`.

The grayscale vector `w` is **not observed data**. It is a transformation rule that tells us how strongly red, green, and blue contribute to one grayscale value.

> 🇪🇸 Usaremos dos conjuntos de datos reales incluidos en librerías estándar:
>
> - una imagen real de microscopía a color de `skimage.data`;
> - el conjunto de 1.797 imágenes reales de dígitos manuscritos de `sklearn`.
>
> El vector de pesos `w` **no es un conjunto de datos observado**. Es una regla de transformación que indica cuánto contribuyen rojo, verde y azul a un valor en escala de grises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# ---------------------------------------------------------------------------
# Real colour images
# ---------------------------------------------------------------------------
photo = data.immunohistochemistry().astype(float)          # (512, 512, 3)
batch = np.stack([
    photo,
    data.astronaut().astype(float),
])                                                         # (2, 512, 512, 3)

# RGB -> grayscale transformation weights.
w = np.array([0.2125, 0.7154, 0.0721])

# ---------------------------------------------------------------------------
# Real handwritten digits
# ---------------------------------------------------------------------------
digits = load_digits()
digit_images = digits.images.astype(float)                 # (1797, 8, 8)

# Tiny matrices for Exercise 2 come from real digit pixels.
A = digit_images[0, 2:4, 2:4]
B = digit_images[1, 2:4, 2:4]

print("Photo / Foto:", photo.shape)
print("Image batch / Lote de imágenes:", batch.shape)
print("Digits / Dígitos:", digit_images.shape)
print("Labels / Etiquetas:", digits.target.shape)
print()
print("A comes from digit label / A proviene del dígito:", int(digits.target[0]))
print(A)
print()
print("B comes from digit label / B proviene del dígito:", int(digits.target[1]))
print(B)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## One rule at three scales / Una regla a tres escalas

Contraction is not a matrix-algebra trick. Reducing three RGB measurements to one grayscale value, multiplying two matrices, and comparing one image vector against thousands are the same operation with different letters: multiply matching positions, add them, and the summed axis disappears.

> 🇪🇸 Una contracción no es un truco de álgebra matricial. Convertir tres valores RGB en un gris, multiplicar dos matrices y comparar un vector de imagen contra miles son la misma operación con otras letras: multiplicar posiciones correspondientes, sumarlas, y el eje sumado desaparece.


### Interactive einsum translator / Traductor interactivo de einsum

Choose an expression. The notebook will tell you:

- which indices enter;
- which index disappears;
- which indices survive;
- what the operation means.

> 🇪🇸 Elige una expresión. El cuaderno te dirá qué índices entran, cuál desaparece, cuáles sobreviven y qué significa la operación.

In [ ]:
einsum_choice = widgets.Dropdown(
    options=[
        ("hwc,c->hw · RGB to gray / RGB a gris", "gray"),
        ("ii-> · Trace / Traza", "trace"),
        ("ij->ji · Transpose / Transpuesta", "transpose"),
        ("ik,kj->ij · Matrix product / Producto matricial", "matmul"),
        ("id,jd->ij · Similarity / Similitud", "similarity"),
    ],
    value="gray",
    description="Expression / Expresión:",
    style={"description_width": "150px"},
)

def explain_einsum_rule(choice):
    explanations = {
        "gray": (
            "hwc,c->hw",
            "c",
            "h,w",
            "multiply RGB values by RGB weights, then sum colour",
            "multiplicar valores RGB por pesos RGB y después sumar color",
        ),
        "trace": (
            "ii->",
            "i",
            "none / ninguno",
            "sum the diagonal to one scalar",
            "sumar la diagonal y obtener un escalar",
        ),
        "transpose": (
            "ij->ji",
            "none / ninguno",
            "j,i",
            "reorder axes; nothing is summed",
            "reordenar ejes; no se suma ningún índice",
        ),
        "matmul": (
            "ik,kj->ij",
            "k",
            "i,j",
            "multiply along the shared k dimension and sum it",
            "multiplicar a lo largo de la dimensión compartida k y sumarla",
        ),
        "similarity": (
            "id,jd->ij",
            "d",
            "i,j",
            "combine d features into one score for each image pair",
            "combinar d características en una puntuación para cada par de imágenes",
        ),
    }

    expr, disappears, survives, en, es = explanations[choice]

    print("Expression / Expresión:", expr)
    print("Disappears / Desaparece:", disappears)
    print("Survives / Permanece:", survives)
    print("EN:", en)
    print("ES:", es)

einsum_rule_output = widgets.interactive_output(
    explain_einsum_rule,
    {"choice": einsum_choice},
)

display(widgets.VBox([einsum_choice, einsum_rule_output]))

## Exercise 1 — contract the colour axis / Ejercicio 1 — contrae el eje de color

`photo` is a real microscopy image:

`photo.shape = (512, 512, 3)`

Read that as:

`(H, W, C) = height × width × colour`

The contraction is:

`hwc,c->hw`

### Predict first / Predice primero

1. Which index disappears?
2. Which indices survive?
3. Why must the result have shape `(512,512)`?

> 🇪🇸 `photo` es una imagen real de microscopía con forma `(512,512,3)`.
>
> En `hwc,c->hw`, predice:
>
> 1. ¿qué índice desaparece?
> 2. ¿qué índices permanecen?
> 3. ¿por qué el resultado debe tener forma `(512,512)`?

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# Use one einsum to convert `photo` to grayscale:
#     'hwc,c->hw'
# Expected shape: (512, 512)
#
# ES:
# Usa un solo einsum para convertir `photo` a gris:
#     'hwc,c->hw'
# Forma esperada: (512, 512)
#
# TODO 2 / TAREA 2
#
# EN:
# Do the same for the entire real image batch in ONE einsum:
#     'nhwc,c->nhw'
# Expected shape: (2, 512, 512)
#
# ES:
# Haz lo mismo para todo el lote real en UN solo einsum:
#     'nhwc,c->nhw'
# Forma esperada: (2, 512, 512)
#
# Explain / Explica:
# - which index is contracted / qué índice se contrae
# - which indices survive / qué índices permanecen

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

gray = np.einsum("hwc,c->hw", photo, w)
gray_batch = np.einsum("nhwc,c->nhw", batch, w)

print("Single image / Imagen individual:")
print(photo.shape, "->", gray.shape)
print()

print("Batch / Lote:")
print(batch.shape, "->", gray_batch.shape)
print()

print("EN: c disappears, so colour is multiplied by weights and summed.")
print("ES: c desaparece, por lo que color se multiplica por pesos y se suma.")
print("EN: h and w survive; n also survives for the batch.")
print("ES: h y w permanecen; n también permanece en el lote.")

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))

axes[0].imshow(photo / 255.0)
axes[0].set_title("Real RGB input / Entrada RGB real")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("hwc,c->hw\nc disappears / c desaparece")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Interactive RGB contraction / Contracción RGB interactiva

Move the red, green, and blue weights.

The real pixels do **not** change. Only the rule for combining the three channels changes.

The bar chart uses the actual RGB colours so you can see which channel is receiving more weight.

> 🇪🇸 Mueve los pesos rojo, verde y azul.
>
> Los píxeles reales **no cambian**. Solo cambia la regla que combina los tres canales.
>
> El gráfico de barras usa los colores RGB reales para mostrar qué canal recibe mayor peso.

In [ ]:
r_slider = widgets.FloatSlider(
    value=float(w[0]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="R / Rojo:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

g_slider = widgets.FloatSlider(
    value=float(w[1]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="G / Verde:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

b_slider = widgets.FloatSlider(
    value=float(w[2]),
    min=0.0,
    max=1.0,
    step=0.025,
    description="B / Azul:",
    readout_format=".3f",
    continuous_update=False,
    style={"description_width": "90px"},
)

def explore_colour_contraction(r, g, b):
    weights = np.array([r, g, b], dtype=float)
    live_gray = np.einsum("hwc,c->hw", photo, weights)

    plt.close("all")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))

    axes[0].imshow(photo / 255.0)
    axes[0].set_title("Same real RGB input / Misma entrada RGB")
    axes[0].axis("off")

    axes[1].imshow(live_gray, cmap="gray")
    axes[1].set_title("Contracted image / Imagen contraída")
    axes[1].axis("off")

    axes[2].bar(
        ["R", "G", "B"],
        weights,
        color=["red", "green", "blue"],
    )
    axes[2].set_ylim(0, 1)
    axes[2].set_title("Weights / Pesos")
    axes[2].set_ylabel("weight / peso")

    plt.tight_layout()
    plt.show()

    print("einsum: hwc,c->hw")
    print(f"Weight sum / Suma de pesos: {weights.sum():.3f}")
    print("EN: c still disappears; only the numeric contribution of R/G/B changed.")
    print("ES: c sigue desapareciendo; solo cambió la contribución numérica de R/G/B.")

rgb_output = widgets.interactive_output(
    explore_colour_contraction,
    {"r": r_slider, "g": g_slider, "b": b_slider},
)

display(
    widgets.VBox([
        widgets.HBox([r_slider, g_slider, b_slider]),
        rgb_output,
    ])
)

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

At every pixel `(h,w)` there are three numbers:

`[R, G, B]`

and three weights:

`[wR, wG, wB]`

`einsum('hwc,c->hw', photo, w)` calculates:

`R×wR + G×wG + B×wB`

for every pixel.

The colour index `c` is no longer needed after the sum, so it disappears.

> 🇪🇸 En cada píxel `(h,w)` existen tres números `[R,G,B]` y tres pesos.
>
> `einsum('hwc,c->hw', photo, w)` calcula una suma ponderada para cada píxel.
>
> Después de sumar sobre color, el índice `c` ya no es necesario y desaparece.

</details>

## Exercise 2 — three matrix operations from real digit pixels / Ejercicio 2 — tres operaciones matriciales con píxeles reales

Matrices `A` and `B` are **not invented toy values**.

They are `2×2` central patches cut from two real `8×8` handwritten digits.

Pixel intensities in this dataset range from `0` to `16`.

We keep the matrices tiny so you can inspect the arithmetic while still using observed data.

### Three expressions / Tres expresiones

**Trace / Traza**

`ii->`

`i` disappears → diagonal values are summed.

**Transpose / Transpuesta**

`ij->ji`

No index disappears → axes are only reordered.

**Matrix product / Producto matricial**

`ik,kj->ij`

`k` disappears → multiply and sum over the shared dimension.

> 🇪🇸 `A` y `B` son recortes centrales `2×2` de dos dígitos manuscritos reales.
>
> La traza suma la diagonal, la transpuesta reordena índices y el producto matricial contrae la dimensión compartida `k`.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# Rewrite each operation with einsum and verify it against NumPy:
#
# a) trace of A           -> scalar
# b) transpose of A       -> (2, 2)
# c) matrix product A @ B -> (2, 2)
#
# ES:
# Reescribe cada operación con einsum y verifícala con NumPy:
#
# a) traza de A           -> escalar
# b) transpuesta de A     -> (2, 2)
# c) producto A @ B       -> (2, 2)
#
# For each one / Para cada una:
# - which index disappears? / ¿qué índice desaparece?
# - which index survives? / ¿qué índice permanece?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

trace_e = np.einsum("ii->", A)
transpose_e = np.einsum("ij->ji", A)
product_e = np.einsum("ik,kj->ij", A, B)

print("A — real digit label / etiqueta real:", int(digits.target[0]))
print(A)
print()
print("B — real digit label / etiqueta real:", int(digits.target[1]))
print(B)
print()

print("Trace / Traza:", trace_e, "| NumPy:", np.trace(A))
print("Transpose / Transpuesta:")
print(transpose_e)
print("Matrix product / Producto matricial:")
print(product_e)

assert np.allclose(trace_e, np.trace(A))
assert np.allclose(transpose_e, A.T)
assert np.allclose(product_e, A @ B)

print()
print("EN: all three einsum results agree with NumPy.")
print("ES: los tres resultados de einsum coinciden con NumPy.")

fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))

for ax, idx, name in zip(
    axes,
    [0, 1],
    ["A", "B"],
):
    ax.imshow(
        digit_images[idx],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    ax.add_patch(
        plt.Rectangle(
            (1.5, 1.5),
            2,
            2,
            fill=False,
            linewidth=2,
        )
    )
    ax.set_title(
        f"{name} · label/etiqueta {int(digits.target[idx])}\n"
        "2×2 real pixel patch / recorte real"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

### Interactive matrix-operation explorer / Explorador interactivo de operaciones matriciales

Choose an operation.

The notebook will show the result and explain the index rule in English and Spanish.

> 🇪🇸 Elige una operación. El cuaderno mostrará el resultado y explicará la regla de índices en inglés y español.

In [ ]:
matrix_operation = widgets.ToggleButtons(
    options=[
        ("Trace / Traza", "trace"),
        ("Transpose / Transpuesta", "transpose"),
        ("Matrix product / Producto", "product"),
    ],
    value="product",
    description="Operation / Operación:",
    style={"description_width": "145px"},
)

def explore_matrix_operation(op):
    if op == "trace":
        result = np.einsum("ii->", A)
        expr = "ii->"
        en = "i is repeated and disappears, so the diagonal is summed."
        es = "i se repite y desaparece, por lo que se suma la diagonal."
        print("A =")
        print(A)

    elif op == "transpose":
        result = np.einsum("ij->ji", A)
        expr = "ij->ji"
        en = "No index disappears; i and j only exchange positions."
        es = "Ningún índice desaparece; i y j solo intercambian posiciones."
        print("A =")
        print(A)

    else:
        result = np.einsum("ik,kj->ij", A, B)
        expr = "ik,kj->ij"
        en = "k disappears; multiply along k and sum, while i and j survive."
        es = "k desaparece; se multiplica y suma sobre k, mientras i y j permanecen."
        print("A =")
        print(A)
        print("B =")
        print(B)

    print()
    print("Expression / Expresión:", expr)
    print("Result / Resultado:")
    print(result)
    print("EN:", en)
    print("ES:", es)

matrix_output = widgets.interactive_output(
    explore_matrix_operation,
    {"op": matrix_operation},
)

display(widgets.VBox([matrix_operation, matrix_output]))

### See one matrix-product cell step by step / Observa una celda del producto paso a paso

For `A @ B`, select an output position `(i,j)`.

The notebook will show exactly which products are added to create that one cell.

> 🇪🇸 Para `A @ B`, selecciona una posición de salida `(i,j)`.
>
> El cuaderno mostrará exactamente qué productos se suman para crear esa celda.

In [ ]:
i_selector = widgets.ToggleButtons(
    options=[0, 1],
    value=0,
    description="i:",
)

j_selector = widgets.ToggleButtons(
    options=[0, 1],
    value=0,
    description="j:",
)

def explain_matmul_cell(i, j):
    products = A[i, :] * B[:, j]
    value = products.sum()

    print(f"Output / Salida C[{i},{j}]")
    print()
    print(
        f"A[{i},0]×B[0,{j}] + A[{i},1]×B[1,{j}]"
    )
    print(
        f"{A[i,0]:g}×{B[0,j]:g} + "
        f"{A[i,1]:g}×{B[1,j]:g}"
    )
    print("=", float(value))
    print()
    print("EN: k takes values 0 and 1, then disappears because those products are summed.")
    print("ES: k toma los valores 0 y 1 y después desaparece porque esos productos se suman.")

matmul_cell_output = widgets.interactive_output(
    explain_matmul_cell,
    {"i": i_selector, "j": j_selector},
)

display(
    widgets.VBox([
        widgets.HBox([i_selector, j_selector]),
        matmul_cell_output,
    ])
)

## Exercise 3 — 3,229,209 similarities from real digit images / Ejercicio 3 — 3.229.209 similitudes entre imágenes reales

The handwritten digits are only `8×8` pixels.

That blocky appearance is **the original measured resolution**, not a bad download.

Each digit therefore has:

`8 × 8 = 64`

real pixel features.

After flattening:

`D.shape = (1797, 64)`

We can compare every image `i` with every image `j` using:

`id,jd->ij`

### Read the indices / Lee los índices

- `i` = query image / imagen consulta → survives / permanece
- `j` = candidate image / imagen candidata → survives / permanece
- `d` = 64 pixel features / 64 características de píxel → disappears / desaparece

So the output is:

`(1797,1797)`

one score for every image pair.

### Raw dot vs cosine / Producto punto vs coseno

Both use the same contraction.

- **Raw dot product** also depends strongly on vector magnitude or total intensity.
- **Cosine similarity** normalizes each image vector first, so it emphasizes the direction/pattern of the 64 pixel values.

> 🇪🇸 Cada dígito tiene 64 características reales de píxel.
>
> En `id,jd->ij`, `d` desaparece y se suman las 64 características; `i` y `j` permanecen.
>
> Por eso obtenemos una matriz `(1797,1797)` con una puntuación para cada par de imágenes.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# Reshape the 1,797 real 8×8 images into:
#     D.shape == (1797, 64)
#
# ES:
# Reorganiza las 1.797 imágenes reales 8×8 para obtener:
#     D.shape == (1797, 64)
#
# TODO 5 / TAREA 5
#
# EN:
# Compute every raw dot-product similarity with ONE einsum:
#     'id,jd->ij'
#
# ES:
# Calcula todas las similitudes de producto punto con UN einsum:
#     'id,jd->ij'
#
# TODO 6 / TAREA 6
#
# EN:
# Normalize every row of D to unit length and repeat the SAME einsum
# to obtain cosine similarity.
#
# ES:
# Normaliza cada fila de D a longitud unitaria y repite el MISMO einsum
# para obtener similitud coseno.
#
# TODO 7 / TAREA 7
#
# EN:
# Use query_idx = 14.
# Exclude self-matching and compare:
# - best raw-dot match
# - top 5 cosine matches
#
# ES:
# Usa query_idx = 14.
# Excluye la coincidencia consigo misma y compara:
# - mejor coincidencia por producto punto
# - 5 mejores coincidencias por coseno
#
# Predict / Predice:
# Which index disappears in 'id,jd->ij'?
# ¿Qué índice desaparece en 'id,jd->ij'?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

D = digit_images.reshape(len(digit_images), -1)

S = np.einsum("id,jd->ij", D, D)

norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)

C = np.einsum("id,jd->ij", Dn, Dn)

assert D.shape == (1797, 64)
assert S.shape == (1797, 1797)
assert C.shape == (1797, 1797)
assert np.allclose(C.diagonal(), 1.0)

print("Flattened digits / Dígitos aplanados:", D.shape)
print("Raw similarity / Similitud producto punto:", S.shape)
print("Cosine similarity / Similitud coseno:", C.shape)
print("Pairwise scores / Puntuaciones por pares:", S.size)
print()
print("EN: d=64 pixel features disappeared; i and j survived.")
print("ES: d=64 características de píxel desapareció; i y j permanecieron.")

query_idx = 14
query_label = int(digits.target[query_idx])

raw_scores = S[query_idx].copy()
cos_scores = C[query_idx].copy()

raw_scores[query_idx] = -np.inf
cos_scores[query_idx] = -np.inf

raw_top1 = int(np.argmax(raw_scores))
cos_top5 = np.argsort(cos_scores)[-5:][::-1]

print()
print("Query / Consulta:", query_idx, "| label/etiqueta:", query_label)
print(
    "Best raw-dot match / Mejor producto punto:",
    raw_top1,
    "| label/etiqueta:",
    int(digits.target[raw_top1]),
)
print(
    "Top-5 cosine indices / Índices top-5 coseno:",
    cos_top5.tolist(),
)
print(
    "Top-5 cosine labels / Etiquetas top-5 coseno:",
    digits.target[cos_top5].astype(int).tolist(),
)

fig, axes = plt.subplots(1, 6, figsize=(11, 2.5))

axes[0].imshow(
    digit_images[query_idx],
    cmap="gray_r",
    interpolation="nearest",
    vmin=0,
    vmax=16,
)
axes[0].set_title(
    f"query / consulta\nlabel {query_label}"
)

for ax, idx in zip(axes[1:], cos_top5):
    ax.imshow(
        digit_images[idx],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    ax.set_title(
        f"label {int(digits.target[idx])}\n"
        f"cos={C[query_idx, idx]:.3f}"
    )

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# The flattened digit matrix and the two similarity matrices (Exercise 3),
# recomputed here in a visible cell so the retrieval and pixel-vector explorers
# below run whether or not the folded solution was executed. The query-14
# ranking comparison, the assertions, and the cosine-vs-dot reasoning stay
# folded in the solution above.
D = digit_images.reshape(len(digit_images), -1)
S = np.einsum("id,jd->ij", D, D)

norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)
C = np.einsum("id,jd->ij", Dn, Dn)

### Interactive digit-retrieval explorer / Explorador interactivo de búsqueda de dígitos

Now turn the mathematics into a small search engine.

Choose:

- any of the 1,797 real digit images;
- **Cosine** or **Raw dot product**;
- how many neighbours to display.

The images remain intentionally pixelated because each visible square is one of the original 64 measurements.

> 🇪🇸 Ahora convierte las matemáticas en un pequeño buscador.
>
> Elige cualquier dígito real, selecciona **Coseno** o **Producto punto** y decide cuántos vecinos mostrar.
>
> Las imágenes se mantienen pixeladas intencionalmente porque cada cuadrado visible corresponde a una de las 64 mediciones originales.

In [ ]:
query_slider = widgets.IntSlider(
    value=14,
    min=0,
    max=len(digit_images) - 1,
    step=1,
    description="Query / Consulta:",
    continuous_update=False,
    style={"description_width": "115px"},
)

similarity_toggle = widgets.ToggleButtons(
    options=[
        ("Cosine / Coseno", "cosine"),
        ("Raw dot / Producto punto", "raw"),
    ],
    value="cosine",
    description="Metric / Métrica:",
    style={"description_width": "100px"},
)

k_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=8,
    step=1,
    description="Top k:",
    continuous_update=False,
    style={"description_width": "55px"},
)

def explore_retrieval(query, metric, k):
    matrix = C if metric == "cosine" else S
    scores = matrix[query].copy()
    scores[query] = -np.inf

    top = np.argsort(scores)[-k:][::-1]
    q_label = int(digits.target[query])

    plt.close("all")
    fig, axes = plt.subplots(
        1,
        k + 1,
        figsize=(2.05 * (k + 1), 2.75),
    )
    axes = np.atleast_1d(axes)

    axes[0].imshow(
        digit_images[query],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    axes[0].set_title(
        f"query {query}\nlabel {q_label}"
    )
    axes[0].axis("off")

    for ax, idx in zip(axes[1:], top):
        ax.imshow(
            digit_images[idx],
            cmap="gray_r",
            interpolation="nearest",
            vmin=0,
            vmax=16,
        )

        score_name = "cos" if metric == "cosine" else "dot"

        ax.set_title(
            f"idx {idx}\n"
            f"label {int(digits.target[idx])}\n"
            f"{score_name}={scores[idx]:.3f}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    labels = digits.target[top].astype(int).tolist()
    same_label = sum(label == q_label for label in labels)

    print("Query label / Etiqueta consulta:", q_label)
    print("Retrieved labels / Etiquetas recuperadas:", labels)
    print(
        f"Same-label matches / Coincidencias misma etiqueta: "
        f"{same_label}/{k}"
    )
    print("einsum: id,jd->ij")
    print("EN: d disappears; i and j survive.")
    print("ES: d desaparece; i y j permanecen.")

    if metric == "cosine":
        print("EN: cosine normalizes magnitude first and emphasizes pixel-pattern direction.")
        print("ES: el coseno normaliza primero la magnitud y enfatiza la dirección del patrón de píxeles.")
    else:
        print("EN: raw dot product also rewards magnitude/intensity.")
        print("ES: el producto punto también favorece magnitud/intensidad.")

retrieval_output = widgets.interactive_output(
    explore_retrieval,
    {
        "query": query_slider,
        "metric": similarity_toggle,
        "k": k_slider,
    },
)

display(
    widgets.VBox([
        query_slider,
        similarity_toggle,
        k_slider,
        retrieval_output,
    ])
)

### See the 64-dimensional vector / Observa el vector de 64 dimensiones

Select a digit and inspect the same data in two forms:

- the original `8×8` image;
- the flattened 64-value vector used by `id,jd->ij`.

Nothing was invented or removed — only the arrangement changed.

> 🇪🇸 Selecciona un dígito y observa los mismos datos en dos formas:
>
> - imagen original `8×8`;
> - vector aplanado de 64 valores usado por `id,jd->ij`.
>
> No se inventó ni eliminó información; solo cambió la organización.

In [ ]:
vector_query = widgets.IntSlider(
    value=14,
    min=0,
    max=len(digit_images) - 1,
    step=1,
    description="Digit / Dígito:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def show_vectorized_digit(query):
    vector = D[query]

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

    axes[0].imshow(
        digit_images[query],
        cmap="gray_r",
        interpolation="nearest",
        vmin=0,
        vmax=16,
    )
    axes[0].set_title(
        f"8×8 image / imagen\nlabel {int(digits.target[query])}"
    )
    axes[0].axis("off")

    axes[1].bar(np.arange(64), vector)
    axes[1].set_title("64 pixel features / 64 características")
    axes[1].set_xlabel("d = pixel feature / característica de píxel")
    axes[1].set_ylabel("intensity / intensidad")

    plt.tight_layout()
    plt.show()

    print("Image shape / Forma imagen:", digit_images[query].shape)
    print("Vector shape / Forma vector:", vector.shape)
    print("EN: d runs from 0 to 63 and is the index contracted in the similarity calculation.")
    print("ES: d recorre 0 a 63 y es el índice que se contrae en el cálculo de similitud.")

vector_output = widgets.interactive_output(
    show_vectorized_digit,
    {"query": vector_query},
)

display(widgets.VBox([vector_query, vector_output]))

<details>
<summary><strong>Why does cosine change the neighbours? / ¿Por qué el coseno cambia los vecinos?</strong></summary>

The raw dot product:

`D_i · D_j`

gets larger when vectors have both:

- similar directions/patterns;
- large magnitudes.

Cosine similarity first divides each vector by its length.

That removes most of the magnitude effect and focuses more on the **pattern of relative pixel intensities**.

The contraction itself remains:

`id,jd->ij`

What changed was the preprocessing.

> 🇪🇸 El producto punto crudo aumenta tanto por similitud de patrón como por magnitud.
>
> La similitud coseno normaliza primero cada vector por su longitud y reduce el efecto de la magnitud.
>
> La contracción sigue siendo `id,jd->ij`; lo que cambió fue el preprocesamiento.

</details>

## What just happened / Qué acaba de pasar

You used **one index rule** at three scales.

### The sentence to remember / La frase para recordar

> **If an index disappears after `->`, it is summed. If it remains, it survives in the output.**

> 🇪🇸
>
> **Si un índice desaparece después de `->`, se suma. Si permanece, sobrevive en la salida.**

Keep this rule for section 10: a longer expression such as:

`ijk,ia,jb,kc->abc`

uses exactly the same logic.

> 🇪🇸 Conserva esta regla para la sección 10: una expresión más larga usa exactamente la misma lógica.


---

## Done with this section / Fin de esta sección

Next / Siguiente: **07 · Inverses and the pseudoinverse / Inversas y la pseudoinversa** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)